In [ ]:
import pymysql
import pandas as pd

# 数据库连接
conn = pymysql.connect(
    host='###',
    port=3306,
    user='###',
    password='###',
    database='ch00',
    charset='utf8mb4'
)

# 设定目标日
target_date = '2025-06-06'

# 步骤1：获取目标日中证500成分股的 ticker 列表
sql_get_tickers = f"""
SELECT ticker
FROM idx_wgt
WHERE idx = '000905' AND dt = '{target_date}'
"""
tickers_df = pd.read_sql(sql_get_tickers, conn)
tickers = tickers_df['ticker'].tolist()
tickers_str = ','.join([f"'{t}'" for t in tickers])  # 用于SQL IN语句

# 步骤2：获取这些股票过去40天的行情数据
sql_price_data = f"""
SELECT dt, ticker, last AS close_price
FROM adjprc
WHERE ticker IN ({tickers_str})
  AND dt <= '{target_date}'
ORDER BY dt DESC
LIMIT {40 * len(tickers)}  -- 估算最大行数,Python中再处理实际交易日
"""
price_df = pd.read_sql(sql_price_data, conn)

# 步骤3：获取目标日的权重
sql_weight_data = f"""
SELECT ticker, wgt
FROM idx_wgt
WHERE idx = '000905' AND dt = '{target_date}'
"""
wgt_df = pd.read_sql(sql_weight_data, conn).set_index('ticker')

# 步骤4：清洗 + 选取最近40个交易日
price_df['dt'] = pd.to_datetime(price_df['dt'])
latest_40_dates = price_df['dt'].drop_duplicates().sort_values(ascending=False).head(40)
price_df = price_df[price_df['dt'].isin(latest_40_dates)]

# 重新整理为价格矩阵
price_matrix = price_df.pivot(index='dt', columns='ticker', values='close_price').sort_index()

# 关闭数据库
conn.close()

# 输出检查
print("价格矩阵：", price_matrix.shape)
print("权重向量：", wgt_df.shape)
print(price_matrix)
print(wgt_df)

In [ ]:
import numpy as np

# Step 1: 计算对数收益率
log_returns = np.log(price_matrix / price_matrix.shift(1)).dropna()

# Step 2: 计算协方差矩阵（用于最小方差优化）
cov_matrix = log_returns.cov()

# Step 3: 计算平均对数收益率（用于目标收益优化）
mean_returns = log_returns.mean()

# 输出检查
print("协方差矩阵维度:", cov_matrix.shape)
print("平均收益率维度:", mean_returns.shape)
print(mean_returns)

In [ ]:
import cvxpy as cp

# 数据准备
mu = mean_returns.values
Sigma = cov_matrix.values
r_f = 0.02 / 252
n = len(mu)
excess_mu = mu - r_f

# 定义变量
x = cp.Variable(n)
t = cp.Variable()  # 代表夏普比率下界

# 构造问题
objective = cp.Maximize(t)
constraints = [
    excess_mu @ x >= t,                # 超额收益 ≥ t
    cp.quad_form(x, Sigma) <= 1,       # 风险归一
    cp.sum(x) == 1,                    # 满仓
    x >= 0                             # 无空头
]

prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.MOSEK)

# 最优权重
opt_weight = x.value

# 转为DataFrame
opt_df = pd.DataFrame({
    'ticker': cov_matrix.columns,
    'sharpe_weight': opt_weight
}).set_index('ticker')

# 合并原始指数权重
combined_df = opt_df.join(wgt_df, how='inner').rename(columns={'wgt': 'index_weight'})
combined_df = combined_df.sort_values(by='sharpe_weight', ascending=False)

# 输出结果
print(combined_df.head(10))


